# NBS site query — heat E2E (Porto Alegre)

Bairro-level heat screening + **250 m grid** intra-bairro differentiation.

Follows [`recommended-datasets.md`](../docs/recommended-datasets.md) and [`heat_nbs_dataset_lens.md`](../docs/heat_nbs_dataset_lens.md).

| Section | Unit | Purpose |
|---------|------|---------|
| **Bairro** | Bairro polygon | Step 0 priority + mechanism + heat NBS |
| **Grid** | 250 m cell | Per-cell heat mechanism type + dominant NBS |

**Default site:** Cidade Baixa — dense central bairro with high heat hazard/risk

**CLI:** `run_e2e.py --hazard heat`

**Setup:** run with cwd = `scripts/`; use nbs_e2e venv or floods `.venv`.


## Setup — paths and imports


In [ ]:
import json
import os
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

# Jupyter kernels often omit Homebrew from PATH on macOS.
for _brew_bin in (Path("/opt/homebrew/bin"), Path("/usr/local/bin")):
    if _brew_bin.is_dir():
        _p = str(_brew_bin)
        if _p not in os.environ.get("PATH", "").split(":"):
            os.environ["PATH"] = f"{_p}:{os.environ.get('PATH', '')}"

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from rasterio.transform import array_bounds
from shapely.geometry import mapping

NOTEBOOK_DIR = Path.cwd()
NBS_E2E_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "scripts" else NOTEBOOK_DIR
SCRIPTS_DIR = NBS_E2E_ROOT / "scripts"
if not (SCRIPTS_DIR / "catalog_layers.py").exists():
    raise FileNotFoundError(
        "Run this notebook with cwd = transformation/nbs_screening/scripts "
        f"(expected catalog_layers.py under {SCRIPTS_DIR})"
    )
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

for mod in ("catalog_layers", "nbs_rules", "grid_screening"):
    sys.modules.pop(mod, None)
import catalog_layers as _catalog_layers
import nbs_rules as _nbs_rules

OUT_DIR = NBS_E2E_ROOT / "output"
IN_DIR = OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

import grid_screening as _grid_screening

HAZARD = "heat"
CATALOG_COGS = _catalog_layers.get_catalog_cogs(HAZARD)
LOCAL_SCREENING_RASTERS = _catalog_layers.get_local_rasters(HAZARD)
barrio_heat_context = _catalog_layers.barrio_heat_context
query_layers = _catalog_layers.query_layers
recommend_all = _nbs_rules.recommend_all
HEAT_NBS_TYPES = _nbs_rules.HEAT_NBS_TYPES
screen_bairro_grid = _grid_screening.screen_bairro_grid
screen_poa_heat_mechanism_grid = _grid_screening.screen_poa_heat_mechanism_grid
cells_to_geodataframe = _grid_screening.cells_to_geodataframe
result_to_geojson = _grid_screening.result_to_geojson
result_to_report_dict = _grid_screening.result_to_report_dict
export_heat_mechanism_geotiff = _grid_screening.export_heat_mechanism_geotiff
export_poa_heat_mechanism_layers = _grid_screening.export_poa_heat_mechanism_layers
MECHANISM_RASTER_NODATA = _grid_screening.MECHANISM_RASTER_NODATA

HEAT_MECHANISM_TYPE_CODES = _nbs_rules.HEAT_MECHANISM_TYPE_CODES
MECHANISM_MIXED_COLOR = _nbs_rules.MECHANISM_MIXED_COLOR
HEAT_MECHANISM_CATALOG_DOCS = _nbs_rules.HEAT_MECHANISM_CATALOG_DOCS
HEAT_MECHANISM_DISPLAY_LABELS = _nbs_rules.HEAT_MECHANISM_DISPLAY_LABELS
HEAT_MIN_STRENGTH = _nbs_rules.HEAT_MIN_STRENGTH
classify_dominant_heat_mechanism = _nbs_rules.classify_dominant_heat_mechanism

HEAT_MECHANISM_COLORS = {
    "without_clear_dominant": "#e0e0e0",
    "none": "#e0e0e0",  # legacy alias for pre-rename exports
    "uhi_built_up": "#d73027",
    "shade_deficit": "#fee08b",
    "high_daytime_lst": "#fc8d59",
    "limited_nocturnal_cooling": "#9970ab",
    "high_social_exposure": "#636363",
    "mixed": MECHANISM_MIXED_COLOR,
}
GRID_LAYER_IDS = {"app_heat_250m", "sample_grid_1km"}

HEAT_SITE_NAME = "Cidade Baixa"  # change to test another bairro
print("NBS E2E root:", NBS_E2E_ROOT)
print("Hazard:", HAZARD)
print("Site:", HEAT_SITE_NAME)
print("Catalog layers:", list(CATALOG_COGS.keys()))
print("Local rasters:", list(LOCAL_SCREENING_RASTERS.keys()))


# heat priority screening

## Step 0 — Heat priority screening

Load the bairro polygon and heat **hazard / risk / exposure / vulnerability** from OEF outputs.

In [ ]:
heat_ctx_row = barrio_heat_context(HEAT_SITE_NAME)
heat_site_geom = heat_ctx_row.pop("geometry")

heat_site_gdf = gpd.GeoDataFrame(
    [{"name": HEAT_SITE_NAME, "geometry": heat_site_geom}], crs="EPSG:4326"
)
heat_site_path = IN_DIR / f"site_{HEAT_SITE_NAME.lower().replace(' ', '_')}.geojson"
heat_site_gdf.to_file(heat_site_path, driver="GeoJSON")

heat_step0 = pd.Series(
    {
        "bairro": HEAT_SITE_NAME,
        "hazard_mean": heat_ctx_row["hazard_mean"],
        "risk_mean": heat_ctx_row["risk_mean"],
        "exposure_score": heat_ctx_row["exposure_score"],
        "vulnerability_score": heat_ctx_row["vulnerability_score"],
    }
)
display(heat_step0.to_frame("value"))
print(f"Site saved → {heat_site_path}")
print("Bounds (lon/lat):", heat_site_geom.bounds)
print("Heat catalog layers:", list(CATALOG_COGS.keys()))
print("Heat local fallbacks:", list(LOCAL_SCREENING_RASTERS.keys()))

## Step 1a — Query heat diagnostic layers

**Target:** mask app/catalog COGs (S3) to the site polygon → zonal mean / median / p90.

**Screening grid:** heat H/E/V/R, GHSL built-up, Dynamic World, MODIS NDVI, Hansen tree cover, LST inputs. Local OEF rasters are fallback if catalog COGs are unavailable.

In [ ]:
heat_layers = query_layers(heat_site_geom, hazard=HAZARD)

heat_rows = []
for layer in heat_layers:
    row = {
        "layer_id": layer.layer_id,
        "status": layer.status,
        "note": layer.note[:80] if layer.note else "",
    }
    for k, v in layer.stats.items():
        row[k] = round(v, 4) if isinstance(v, float) else v
    heat_rows.append(row)

heat_layers_df = pd.DataFrame(heat_rows)
display(heat_layers_df)

### Inspect heat screening metrics (inside bairro)

These proxies drive heat mechanism inference: built-up / vegetation, LST norms, and H/E/V/R from catalog COGs (with local fallback).

In [ ]:
heat_grid_layer = next(l for l in heat_layers if l.layer_id in GRID_LAYER_IDS)
heat_water_layer = next(l for l in heat_layers if l.layer_id == "osm_waterways")

heat_grid_stats = heat_grid_layer.stats
heat_water_stats = {**heat_water_layer.stats, "_note": heat_water_layer.note}

print("Screening pixels in site:", heat_grid_stats.get("n_cells"))
print("Source:", heat_grid_layer.note)
display(
    pd.Series({k: v for k, v in heat_grid_stats.items() if k != "n_cells"}).to_frame("mean")
)
print("Waterways:", heat_water_stats)

## Step 1b — Infer heat exposure context

Before choosing a cooling NBS: *what kind of heat problem* dominates here? (UHI, shade deficit, daytime LST, nocturnal retention, social exposure)

In [ ]:
heat_ctx = {"bairro": HEAT_SITE_NAME, "hazard": HAZARD, **heat_ctx_row}
heat_catalog = CATALOG_COGS
for layer in heat_layers:
    if layer.status == "ok" and layer.layer_id in heat_catalog:
        heat_ctx[f"{layer.layer_id}_mean"] = layer.stats.get("mean")
    elif layer.status == "ok" and layer.layer_id == "cougar_heat_hazard_250m":
        heat_ctx["local_heat_hazard_mean"] = layer.stats.get("mean")
    elif layer.status == "ok" and layer.layer_id == "cougar_heat_risk_250m":
        heat_ctx["local_heat_risk_mean"] = layer.stats.get("mean")

heat_mechanism, heat_recommendations = recommend_all(
    heat_ctx, heat_grid_stats, heat_water_stats, hazard=HAZARD
)

heat_mech_df = pd.DataFrame(
    {
        "signal": [
            "uhi_built_up",
            "shade_deficit",
            "high_daytime_lst",
            "limited_nocturnal_cooling",
            "high_social_exposure",
        ],
        "value": [
            heat_mechanism.uhi_built_up,
            heat_mechanism.shade_deficit,
            heat_mechanism.high_daytime_lst,
            heat_mechanism.limited_nocturnal_cooling,
            heat_mechanism.high_social_exposure,
        ],
    }
)
display(heat_mech_df)
print("\nRationale:")
for line in heat_mechanism.rationale:
    print(" •", line)

## Step 2 — Heat NBS typology screening

Simple rule scores (0–1) per cooling NBS type. **≥ 0.55 = plausible** for early ideation only — not microclimate or engineering design.

In [ ]:
heat_recs_df = pd.DataFrame([asdict(r) for r in heat_recommendations])
heat_recs_df["gaps"] = heat_recs_df["gaps"].apply(lambda g: "; ".join(g) if g else "")
display(heat_recs_df[["nbs_type", "score", "rationale", "gaps"]])

print("\nTop 3 for ideation:")
for _, row in heat_recs_df.head(3).iterrows():
    print(f"  [{row['score']:.2f}] {row['nbs_type']}")

## Step 6 — Heat gaps discovered

Catalog errors + rule-level data needs before site design.

In [ ]:
heat_gaps = []
for layer in heat_layers:
    if layer.status == "error":
        heat_gaps.append(f"[{layer.layer_id}] {layer.note}")
for rec in heat_recommendations:
    heat_gaps.extend(rec.gaps)
heat_gaps = sorted(set(heat_gaps))

for i, g in enumerate(heat_gaps, 1):
    print(f"{i}. {g}")

## Save heat report (JSON)

In [ ]:
heat_report = {
    "exercise": "nbs_site_query_heat_e2e",
    "hazard": HAZARD,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "site": {"name": HEAT_SITE_NAME, "geojson": str(heat_site_path)},
    "step_0_priority": heat_ctx,
    "step_1_layers": [
        {
            "layer_id": l.layer_id,
            "source": l.source,
            "status": l.status,
            "stats": l.stats,
            "note": l.note,
        }
        for l in heat_layers
    ],
    "step_1_mechanism": {
        "uhi_built_up": heat_mechanism.uhi_built_up,
        "shade_deficit": heat_mechanism.shade_deficit,
        "high_daytime_lst": heat_mechanism.high_daytime_lst,
        "limited_nocturnal_cooling": heat_mechanism.limited_nocturnal_cooling,
        "high_social_exposure": heat_mechanism.high_social_exposure,
        "rationale": heat_mechanism.rationale,
    },
    "step_2_nbs_recommendations": [asdict(r) for r in heat_recommendations],
    "gaps_discovered": heat_gaps,
}

heat_out_path = OUT_DIR / f"nbs_site_query_heat_{HEAT_SITE_NAME.lower().replace(' ', '_')}.json"
heat_out_path.write_text(json.dumps(heat_report, indent=2, ensure_ascii=False))
print(f"Report written → {heat_out_path}")

# Grid screening — heat mechanism type layer (ON-5991)

**Primary unit = 250 m grid cell**, not bairro average. `HEAT_SITE_NAME` filters which cells to screen.

| Field | Description |
|-------|-------------|
| `heat_mechanism_type` | **Dominant** type: `uhi_built_up` · `shade_deficit` · `high_daytime_lst` · `limited_nocturnal_cooling` · `high_social_exposure` · `mixed` · `without_clear_dominant` |
| `heat_mechanism_code` | Integer code for catalog raster (0–6) |
| Boolean flags | Five mechanism signals (can overlap; type picks dominant by strength) |
| `dominant_nbs` | Top-scoring cooling NBS typology for that cell |

Logic: `nbs_rules.classify_dominant_heat_mechanism()`.

In [ ]:
print(f"Screening 250 m grid cells for {HEAT_SITE_NAME}...", flush=True)
heat_grid_result = screen_bairro_grid(HEAT_SITE_NAME, hazard="heat", sample_catalog=True)
heat_grid_gdf = cells_to_geodataframe(heat_grid_result)

print(f"Cells screened: {heat_grid_result.n_cells} @ ~{heat_grid_result.cell_size_m:.0f} m")
display(pd.Series(heat_grid_result.mechanism_summary).to_frame("value"))

if "dominant_mechanism_type_counts" in heat_grid_result.mechanism_summary:
    display(
        pd.Series(heat_grid_result.mechanism_summary["dominant_mechanism_type_counts"])
        .sort_values(ascending=False)
        .to_frame("cells")
    )

display(
    heat_grid_gdf[
        [
            "cell_id",
            "heat_mechanism_type",
            "heat_mechanism_code",
            "uhi_built_up",
            "shade_deficit",
            "high_daytime_lst",
            "limited_nocturnal_cooling",
            "high_social_exposure",
            "heat_score",
            "dominant_nbs",
        ]
    ]
)

### Map — dominant `heat_mechanism_type` per cell


In [ ]:
HEAT_MECHANISM_COLORS = {
    "without_clear_dominant": "#e0e0e0",
    "none": "#e0e0e0",
    "uhi_built_up": "#d73027",
    "shade_deficit": "#fee08b",
    "high_daytime_lst": "#fc8d59",
    "limited_nocturnal_cooling": "#9970ab",
    "high_social_exposure": "#636363",
    "mixed": MECHANISM_MIXED_COLOR,
}

heat_grid_gdf = heat_grid_gdf.copy()
heat_grid_gdf["color"] = heat_grid_gdf["heat_mechanism_type"].map(HEAT_MECHANISM_COLORS).fillna("#cccccc")

fig, ax = plt.subplots(figsize=(9, 8))
heat_site_gdf.boundary.plot(ax=ax, color="black", linewidth=1.5, label=HEAT_SITE_NAME)
heat_grid_gdf.plot(ax=ax, color=heat_grid_gdf["color"], edgecolor="white", linewidth=0.3, alpha=0.9)
ax.set_title(f"Dominant heat mechanism type — {HEAT_SITE_NAME} (250 m)")

from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=c, edgecolor="white", label=HEAT_MECHANISM_DISPLAY_LABELS.get(t, t))
    for t, c in HEAT_MECHANISM_COLORS.items()
    if t != "none"
]
ax.legend(handles=legend_handles, loc="lower left", fontsize=8, title="heat_mechanism_type")
plt.tight_layout()
plt.show()

### Save grid outputs


In [ ]:
heat_grid_slug = HEAT_SITE_NAME.lower().replace(" ", "_")
heat_grid_geojson_path = OUT_DIR / f"nbs_grid_heat_{heat_grid_slug}.geojson"
heat_grid_json_path = OUT_DIR / f"nbs_grid_heat_{heat_grid_slug}.json"
heat_grid_tif_path = OUT_DIR / f"heat_mechanism_type_{heat_grid_slug}_250m.tif"

heat_geojson_payload = result_to_geojson(heat_grid_result)
heat_geojson_payload["properties"]["layer"] = "poa_heat_mechanism_type"
heat_geojson_payload["properties"]["mechanism_type_codes"] = HEAT_MECHANISM_TYPE_CODES
heat_grid_geojson_path.write_text(json.dumps(heat_geojson_payload, indent=2, ensure_ascii=False))
heat_grid_json_path.write_text(json.dumps(result_to_report_dict(heat_grid_result), indent=2, ensure_ascii=False))
export_heat_mechanism_geotiff(heat_grid_result, heat_grid_tif_path)

print(f"GeoJSON → {heat_grid_geojson_path}")
print(f"Report  → {heat_grid_json_path}")
print(f"GeoTIFF → {heat_grid_tif_path}")

### Optional — full POA heat mechanism layer

Set `BUILD_POA_HEAT_LAYER = True` to build the city-wide heat mechanism raster (~19k hazard-valid pixels; **~5–15 min** with layer preload).

**Methodology, assumptions, and limitations:** [`docs/poa_mechanism_type_layer.md`](../docs/poa_mechanism_type_layer.md)

**Outputs (cascade, like flood hazard base + IDW):**

| File | Role |
|---|---|
| `heat_mechanism_type_poa_250m_observed.tif` | Direct screening on hazard-valid pixels |
| `heat_mechanism_type_poa_250m.tif` | **Filled** layer for tiles (observed + IDW in gaps) |
| `heat_mechanism_is_interpolated_poa_250m.tif` | Mask: 1 = IDW-filled pixel |
| `heat_mechanism_type_poa_250m.geojson` | Hazard-valid screened cells (+ `hazard_valid`, `is_interpolated`) |

Note: the heat hazard COG currently has **full POA coverage**, so IDW usually adds few or zero pixels — but the pipeline matches flood for consistency.

Run the **interactive POA map** cell below (`filled` GeoJSON / TIF), then **`PUBLISH_POA_HEAT_COG_TILES = True`** to build COG + XYZ tiles and upload **COG, tiles, and GeoJSON** to S3 (`UPLOAD_POA_TO_S3`).


In [ ]:
BUILD_POA_HEAT_LAYER = True

if BUILD_POA_HEAT_LAYER:
    poa_heat_result = screen_poa_heat_mechanism_grid(sample_catalog=True, include_nbs=False)
    poa_heat_paths = export_poa_heat_mechanism_layers(poa_heat_result, OUT_DIR)
    poa_heat_tif = poa_heat_paths["filled"]
    poa_heat_observed_tif = poa_heat_paths["observed"]
    poa_heat_interp_tif = poa_heat_paths["is_interpolated"]
    poa_heat_geojson = OUT_DIR / "heat_mechanism_type_poa_250m.geojson"
    poa_heat_payload = result_to_geojson(poa_heat_result)
    poa_heat_payload["properties"]["layer"] = "poa_heat_mechanism_type"
    poa_heat_payload["properties"]["mechanism_type_codes"] = HEAT_MECHANISM_TYPE_CODES
    poa_heat_payload["properties"]["raster_products"] = {
        "observed": poa_heat_observed_tif.name,
        "filled": poa_heat_tif.name,
        "is_interpolated": poa_heat_interp_tif.name,
    }
    poa_heat_payload["properties"]["mechanism_summary"] = {
        **poa_heat_result.mechanism_summary,
        "hazard_valid_cell_count": sum(1 for c in poa_heat_result.cells if c.hazard_valid),
        "interpolated_cell_count": sum(1 for c in poa_heat_result.cells if c.is_interpolated),
    }
    poa_heat_geojson.write_text(json.dumps(poa_heat_payload, ensure_ascii=False))
    print(f"POA heat cells screened: {poa_heat_result.n_cells}")
    print(f"POA observed TIF → {poa_heat_observed_tif}")
    print(f"POA filled TIF   → {poa_heat_tif}")
    print(f"POA interp mask  → {poa_heat_interp_tif}")
    print(f"POA GeoJSON      → {poa_heat_geojson}")
    display(
        pd.Series(poa_heat_result.mechanism_summary.get("dominant_mechanism_type_counts", {}))
        .to_frame("cells")
    )
else:
    print("Skipping full POA heat build (BUILD_POA_HEAT_LAYER=False).")

### Map — full POA `heat_mechanism_type`

Visualizes `output/heat_mechanism_type_poa_250m.tif` (preferred) or `.geojson` after `BUILD_POA_HEAT_LAYER = True`, or from a prior export. The dashed outline is the current `HEAT_SITE_NAME` bairro for context.


In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from rasterio.plot import plotting_extent

if "MECHANISM_MIXED_COLOR" not in globals():
    MECHANISM_MIXED_COLOR = "#8e0152"

if "HEAT_MECHANISM_COLORS" not in globals():
    HEAT_MECHANISM_COLORS = {
        "without_clear_dominant": "#e0e0e0",
        "none": "#e0e0e0",
        "uhi_built_up": "#d73027",
        "shade_deficit": "#fee08b",
        "high_daytime_lst": "#fc8d59",
        "limited_nocturnal_cooling": "#9970ab",
        "high_social_exposure": "#636363",
        "mixed": MECHANISM_MIXED_COLOR,
    }
if "HEAT_MECHANISM_DISPLAY_LABELS" not in globals():
    HEAT_MECHANISM_DISPLAY_LABELS = {"without_clear_dominant": "without a clear dominant mechanism"}

POA_HEAT_TIF = OUT_DIR / "heat_mechanism_type_poa_250m.tif"  # filled (observed + IDW)
POA_HEAT_OBSERVED_TIF = OUT_DIR / "heat_mechanism_type_poa_250m_observed.tif"
POA_HEAT_GEOJSON = OUT_DIR / "heat_mechanism_type_poa_250m.geojson"

HEAT_TYPE_ORDER = [
    "without_clear_dominant",
    "uhi_built_up",
    "shade_deficit",
    "high_daytime_lst",
    "limited_nocturnal_cooling",
    "high_social_exposure",
    "mixed",
]
POA_HEAT_CMAP = ListedColormap([HEAT_MECHANISM_COLORS[t] for t in HEAT_TYPE_ORDER])
POA_HEAT_NORM = BoundaryNorm(np.arange(-0.5, len(HEAT_TYPE_ORDER) + 0.5, 1), POA_HEAT_CMAP.N)

fig, ax = plt.subplots(figsize=(11, 10))
source_label = None

if POA_HEAT_TIF.exists():
    with rasterio.open(POA_HEAT_TIF) as src:
        data = src.read(1, masked=True)
        # Export uses MECHANISM_RASTER_NODATA=255; code 0 = without_clear_dominant is valid.
        if src.nodata != MECHANISM_RASTER_NODATA:
            data = np.ma.masked_equal(data, MECHANISM_RASTER_NODATA)
        ax.imshow(
            data,
            extent=plotting_extent(src),
            origin="upper",
            cmap=POA_HEAT_CMAP,
            norm=POA_HEAT_NORM,
            interpolation="nearest",
        )
    source_label = POA_HEAT_TIF.name
elif POA_HEAT_GEOJSON.exists():
    poa_heat_gdf = gpd.read_file(POA_HEAT_GEOJSON)
    poa_heat_gdf = poa_heat_gdf.copy()
    poa_heat_gdf["color"] = (
        poa_heat_gdf["heat_mechanism_type"].map(HEAT_MECHANISM_COLORS).fillna("#cccccc")
    )
    poa_heat_gdf.plot(ax=ax, color=poa_heat_gdf["color"], edgecolor="none", linewidth=0, alpha=0.95)
    source_label = POA_HEAT_GEOJSON.name
elif "poa_heat_result" in globals():
    poa_heat_gdf = cells_to_geodataframe(poa_heat_result)
    poa_heat_gdf = poa_heat_gdf.copy()
    poa_heat_gdf["color"] = (
        poa_heat_gdf["heat_mechanism_type"].map(HEAT_MECHANISM_COLORS).fillna("#cccccc")
    )
    poa_heat_gdf.plot(ax=ax, color=poa_heat_gdf["color"], edgecolor="none", linewidth=0, alpha=0.95)
    source_label = "in-memory poa_heat_result"
else:
    print(
        "No POA heat layer found. Set BUILD_POA_HEAT_LAYER=True in the cell above, "
        "or ensure output/heat_mechanism_type_poa_250m.tif|.geojson exists."
    )

if source_label:
    try:
        heat_site_gdf.boundary.plot(
            ax=ax,
            color="black",
            linewidth=2,
            linestyle="--",
            label=HEAT_SITE_NAME,
        )
    except NameError:
        site_name = globals().get("HEAT_SITE_NAME", "Cidade Baixa")
        site_path = IN_DIR / f"site_{site_name.lower().replace(' ', '_')}.geojson"
        if site_path.exists():
            gpd.read_file(site_path).boundary.plot(
                ax=ax,
                color="black",
                linewidth=2,
                linestyle="--",
                label=site_name,
            )

    ax.set_title(
        f"Dominant heat mechanism type — Porto Alegre (250 m, filled)\nsource: {source_label}"
    )
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    legend_handles = [
        Patch(facecolor=HEAT_MECHANISM_COLORS[t], edgecolor="white", label=HEAT_MECHANISM_DISPLAY_LABELS.get(t, t))
        for t in HEAT_TYPE_ORDER
    ]
    ax.legend(handles=legend_handles, loc="lower left", fontsize=8, title="heat_mechanism_type")
    plt.tight_layout()
    plt.show()

    if POA_HEAT_GEOJSON.exists():
        summary = json.loads(POA_HEAT_GEOJSON.read_text()).get("properties", {}).get(
            "mechanism_summary", {}
        )
        counts = summary.get("dominant_mechanism_type_counts")
        if counts:
            print("POA dominant heat mechanism counts:")
            display(pd.Series(counts).sort_values(ascending=False).to_frame("cells"))


### Publish POA heat layer — COG + map tiles

Converts the **filled** `heat_mechanism_type_poa_250m.tif` (observed + IDW gap-fill) to **EPSG:3857 COG**, builds **visual** and **value** XYZ tiles (`z=8–15`), and writes the GDAL **colors** file. Requires GDAL CLI (`gdalwarp`, `gdal_translate`, `gdaldem`, `gdal_calc.py`, `gdal2tiles.py`). Run after `BUILD_POA_HEAT_LAYER = True`.

In [ ]:
import os
import shutil
import subprocess

PUBLISH_POA_HEAT_COG_TILES = True  # flip True after BUILD_POA_HEAT_LAYER produces the .tif
UPLOAD_POA_TO_S3 = True  # requires AWS CLI (aws configure)

POA_HEAT_LAYER_SLUG = "heat_mechanism_type_poa_250m"
IN_HEAT_TIF = OUT_DIR / f"{POA_HEAT_LAYER_SLUG}.tif"
POA_HEAT_PUBLISH_DIR = OUT_DIR / POA_HEAT_LAYER_SLUG
HEAT_COLORS_TXT = POA_HEAT_PUBLISH_DIR / f"{POA_HEAT_LAYER_SLUG}_colors.txt"
HEAT_WARPED_TIF = POA_HEAT_PUBLISH_DIR / f"{POA_HEAT_LAYER_SLUG}_3857.tif"
HEAT_COG_TIF = POA_HEAT_PUBLISH_DIR / f"{POA_HEAT_LAYER_SLUG}_cog.tif"
HEAT_COLORIZED_TIF = POA_HEAT_PUBLISH_DIR / f"{POA_HEAT_LAYER_SLUG}_colorized.tif"
HEAT_VALUE_RGB_TIF = POA_HEAT_PUBLISH_DIR / f"{POA_HEAT_LAYER_SLUG}_value_encoded_rgb.tif"
HEAT_VISUAL_TILES_DIR = POA_HEAT_PUBLISH_DIR / "tiles_visual"
HEAT_VALUE_TILES_DIR = POA_HEAT_PUBLISH_DIR / "tiles_values"
HEAT_VALUE_DECODE_TXT = POA_HEAT_PUBLISH_DIR / f"{POA_HEAT_LAYER_SLUG}_value_tiles_decode.txt"

_GDAL_CLI = (
    "gdalwarp",
    "gdal_translate",
    "gdaldem",
    "gdal_calc.py",
    "gdal2tiles.py",
)


def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    h = hex_color.lstrip("#")
    return tuple(int(h[i : i + 2], 16) for i in (0, 2, 4))


def _prepend_common_bin_to_path() -> None:
    for prefix in (Path("/opt/homebrew/bin"), Path("/usr/local/bin")):
        if prefix.is_dir():
            p = str(prefix)
            if p not in os.environ.get("PATH", "").split(":"):
                os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"


def _require_gdal_cli() -> None:
    _prepend_common_bin_to_path()
    missing = [cmd for cmd in _GDAL_CLI if not shutil.which(cmd)]
    if missing:
        raise RuntimeError(
            f"Missing GDAL CLI tools: {missing}. "
            "Install GDAL (e.g. brew install gdal) and restart the kernel, "
            "or ensure /opt/homebrew/bin is on PATH."
        )


if not PUBLISH_POA_HEAT_COG_TILES:
    print(
        "Skipping POA heat COG/tiles publish (PUBLISH_POA_HEAT_COG_TILES=False). "
        "Set True after heat_mechanism_type_poa_250m.tif exists."
    )
elif not IN_HEAT_TIF.exists():
    raise FileNotFoundError(
        f"Missing input raster: {IN_HEAT_TIF}. Run BUILD_POA_HEAT_LAYER=True first."
    )
else:
    _require_gdal_cli()
    POA_HEAT_PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

    color_lines = [
        "# Dominant heat mechanism type (ON-5991). GDAL color-relief for visual tiles.",
        "# Codes from HEAT_MECHANISM_TYPE_CODES in nbs_rules.py",
        f"# Raster nodata = {MECHANISM_RASTER_NODATA} (outside screened grid); code 0 = without_clear_dominant",
        "nv 0 0 0 0",
    ]
    for mech_type, code in sorted(HEAT_MECHANISM_TYPE_CODES.items(), key=lambda kv: kv[1]):
        r, g, b = _hex_to_rgb(HEAT_MECHANISM_COLORS[mech_type])
        color_lines.append(f"{code} {r} {g} {b}")
    HEAT_COLORS_TXT.write_text("\n".join(color_lines) + "\n", encoding="utf-8")
    print("Wrote colors:", HEAT_COLORS_TXT)

    subprocess.run(
        ["gdalwarp", "-t_srs", "EPSG:3857", "-r", "near", "-overwrite", str(IN_HEAT_TIF), str(HEAT_WARPED_TIF)],
        check=True,
    )
    subprocess.run(
        [
            "gdal_translate",
            str(HEAT_WARPED_TIF),
            str(HEAT_COG_TIF),
            "-of",
            "COG",
            "-ot",
            "Byte",
            "-co",
            "COMPRESS=DEFLATE",
            "-co",
            "RESAMPLING=NEAREST",
            "-co",
            "OVERVIEWS=AUTO",
        ],
        check=True,
    )
    print("Created COG:", HEAT_COG_TIF)

    subprocess.run(
        [
            "gdaldem",
            "color-relief",
            "-nearest_color_entry",
            str(HEAT_COG_TIF),
            str(HEAT_COLORS_TXT),
            str(HEAT_COLORIZED_TIF),
            "-alpha",
        ],
        check=True,
    )
    print("Created colorized raster:", HEAT_COLORIZED_TIF)

    HEAT_VISUAL_TILES_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "gdal2tiles.py",
            "-r",
            "near",
            "-z",
            "8-15",
            "--xyz",
            "-w",
            "none",
            str(HEAT_COLORIZED_TIF),
            str(HEAT_VISUAL_TILES_DIR),
        ],
        check=True,
    )
    print("Visual tiles:", HEAT_VISUAL_TILES_DIR)

    base_expr = (
        "numpy.where(numpy.isnan(A), 0, "
        "numpy.rint(numpy.clip(A,0,16777214)).astype(numpy.int64) + 1)"
    )
    subprocess.run(
        [
            "gdal_calc.py",
            "-A",
            str(HEAT_COG_TIF),
            "--calc",
            f"bitwise_and({base_expr},255)",
            "--calc",
            f"bitwise_and(right_shift({base_expr},8),255)",
            "--calc",
            f"bitwise_and(right_shift({base_expr},16),255)",
            "--type",
            "Byte",
            "--NoDataValue",
            "0",
            "--overwrite",
            "--outfile",
            str(HEAT_VALUE_RGB_TIF),
        ],
        check=True,
    )

    HEAT_VALUE_TILES_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "gdal2tiles.py",
            "-r",
            "near",
            "-z",
            "8-15",
            "--xyz",
            "-w",
            "none",
            str(HEAT_VALUE_RGB_TIF),
            str(HEAT_VALUE_TILES_DIR),
        ],
        check=True,
    )
    print("Value tiles:", HEAT_VALUE_TILES_DIR)

    HEAT_VALUE_DECODE_TXT.write_text(
        "\n".join(
            [
                "Heat mechanism type POA value tiles",
                "",
                f"Source raster: {IN_HEAT_TIF}",
                f"COG (EPSG:3857): {HEAT_COG_TIF}",
                f"Visual tiles: {HEAT_VISUAL_TILES_DIR}/{{z}}/{{x}}/{{y}}.png",
                f"Value tiles: {HEAT_VALUE_TILES_DIR}/{{z}}/{{x}}/{{y}}.png",
                "",
                "Value tile encoding (Terrain RGB style):",
                "encoded = R + 256 * G + 65536 * B",
                "if encoded == 0: nodata",
                "else: heat_mechanism_code = encoded - 1",
                "",
                "Mechanism codes:",
                *[
                    f"  {code}: {mech_type}"
                    for mech_type, code in sorted(
                        HEAT_MECHANISM_TYPE_CODES.items(), key=lambda kv: kv[1]
                    )
                ],
            ]
        )
        + "\n",
        encoding="utf-8",
    )
    print("Value decode notes:", HEAT_VALUE_DECODE_TXT)

    if UPLOAD_POA_TO_S3:
        import poa_mechanism_publish as _poa_publish

        _poa_publish.upload_poa_mechanism_to_s3(
            "heat",
            OUT_DIR,
            publish_dir=POA_HEAT_PUBLISH_DIR,
            geojson_path=OUT_DIR / f"{POA_HEAT_LAYER_SLUG}.geojson",
        )
    else:
        print("Skipping S3 upload (UPLOAD_POA_TO_S3=False).")

---

### Try another site

Change `HEAT_SITE_NAME` in Setup.

```bash
geospatial-data/floods/.venv/bin/python transformation/nbs_screening/scripts/run_e2e.py --hazard heat
```
